# Chroma Vector Store


[Step 8 - Chroma]

> **MLCourse - Agentic AI - Vector Stores**
> Stage in the capstone: STORE + RETRIEVE backbone.

# What you will learn

1. Turning Alice chunks (module 06 skills) into vectors inside a Chroma store.
2. Why `persist_directory` makes your index survive kernel restarts.
3. Reopening a persisted store - and why the embedder must match.
4. Scored search: what `similarity_search_with_score` returns (DISTANCES!).
5. Metadata filtering: harvest chapter headings once, query one chapter forever.

Chroma is the course default vector store: an embedded database that runs
in-process, auto-persists to disk, and supports rich metadata filters - all
without any server, container, or API key.

In [1]:
# ---------------------------------------------------------------------------
# Setup cell (identical in every MLCourse notebook): imports, TRACK walker,
# DATA folder creation, .env loading, matplotlib inline magic - guarded so
# the file also runs as a plain script outside Jupyter.
# ---------------------------------------------------------------------------
from pathlib import Path


def find_track(start: Path, target: str = "03_agentic_ai") -> Path:
    """Climb parent folders until a directory named ``target`` shows up."""
    for candidate in [start, *start.parents]:
        probe = candidate / target
        if probe.is_dir():
            return probe
    raise FileNotFoundError(
        f"Could not find '{target}' above {start}. "
        "Open this notebook from inside the MLCourse repository."
    )


TRACK = find_track(Path.cwd())       # .../MLCourse/03_agentic_ai
DATA = TRACK / "data"                # one shared data folder for the track
DATA.mkdir(exist_ok=True)            # no-op when it already exists

from dotenv import load_dotenv       # noqa: E402  reads KEY=value files

load_dotenv()                        # .env beside the current directory
load_dotenv(TRACK / ".env")          # .env at the track root

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass                             # magic only exists inside IPython/Jupyter

print("[setup] TRACK:", TRACK)
print("[setup] DATA :", DATA)

[setup] TRACK: D:\projects\python\MLCourse\03_agentic_ai
[setup] DATA : D:\projects\python\MLCourse\03_agentic_ai\data


### From book to tagged chunks

We slice alice.txt to its first 20000 characters so embedding stays snappy.
Real books carry structure - chapter headings - which is FREE metadata if you
harvest it at ingest time. In this text, genuine headings sit at the very
start of a line ("CHAPTER I.") while the table of contents indents its
entries, so an anchored regex separates signal from boilerplate. Every chunk
then inherits its chapter number, which powers the filter demo later.

In [2]:
import re
import shutil
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

ALICE_URL = "https://www.gutenberg.org/files/11/11-0.txt"


def ensure_alice(data_dir: Path) -> Path:
    """Download alice.txt ONCE into the shared data folder; reuse forever."""
    path = data_dir / "alice.txt"
    if path.exists():
        print(f"[data] cached {path.name}: {path.stat().st_size:,} bytes")
    else:
        import urllib.request
        print("[data] downloading alice.txt (one time) ...")
        urllib.request.urlretrieve(ALICE_URL, path)
        print(f"[data] saved   {path.name}: {path.stat().st_size:,} bytes")
    return path


RAW = ensure_alice(DATA).read_text(encoding="utf-8-sig")[:20_000]  # speed slice

# Anchored regex: ONLY true headings (column 0); indented TOC lines ignored.
marks = list(re.finditer(r"^CHAPTER [IVX]+\.", RAW, flags=re.MULTILINE))
print(f"[data] real chapter headings found in slice: {len(marks)}")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
documents = []
for i, mark in enumerate(marks):
    seg_end = marks[i + 1].start() if i + 1 < len(marks) else len(RAW)
    chapter = str(i + 1)                       # '1', '2', ... string keys for filters
    for piece in splitter.split_text(RAW[mark.start():seg_end]):
        documents.append(Document(
            page_content=piece,
            metadata={"source": "alice", "chapter": chapter},
        ))

counts = {}
for doc in documents:
    counts[doc.metadata["chapter"]] = counts.get(doc.metadata["chapter"], 0) + 1
print("[data] chunks per chapter:", counts)
print("[data] total chunks      :", len(documents))
print("[data] sample metadata   :", documents[0].metadata)

[data] cached alice.txt: 151,191 bytes
[data] real chapter headings found in slice: 2
[data] chunks per chapter: {'1': 35, '2': 25}
[data] total chunks      : 60
[data] sample metadata   : {'source': 'alice', 'chapter': '1'}


### Building the store: from_documents does three jobs at once

One call (1) embeds every chunk via the HuggingFace model, (2) writes vectors
AND documents into Chroma, (3) persists everything under `persist_directory`.
The `collection_name` namespaces data inside the database so unrelated
projects can share one folder safely.

> **Bold caveat**: rerunning `from_documents` against an existing persisted
collection INSERTS DUPLICATES. We wipe the folder first so reruns stay clean -
in production you would upsert with explicit ids instead.

In [3]:
from langchain_chroma import Chroma                     # official wrapper
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # free, local, 384 dims
CHROMA_DIR = DATA / "chroma_alice"

if CHROMA_DIR.exists():          # clean slate: reruns must not double-count
    shutil.rmtree(CHROMA_DIR)
    print("[chroma] removed stale", CHROMA_DIR.name)

vectordb = Chroma.from_documents(
    documents=documents,                                # chunks WITH metadata
    embedding=HuggingFaceEmbeddings(model_name=EMBED_MODEL),
    collection_name="alice",                            # namespace inside DB
    persist_directory=str(CHROMA_DIR),                  # <- THIS makes it durable
)
_n_vecs = len(vectordb.get()["ids"])                   # v1.x: count via collection ids
print("[chroma] vectors in store:", _n_vecs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[chroma] vectors in store: 60


### Proving persistence: reopen from disk

Everything above wrote SQLite files under chroma_alice/. A fresh Chroma
instance pointed at the SAME path + collection reads them back - no re-adding.
This is exactly what happens when tomorrow's process (or the capstone app)
loads the knowledge base built today.

> **Bold rule**: pass the SAME embedding model you indexed with. Vectors and
queries must live in the same space, or results are confidently wrong.

### REOPEN: constructor-only access to the existing on-disk store.


In [ ]:
reopened = Chroma(
    collection_name="alice",
    persist_directory=str(CHROMA_DIR),
    embedding_function=HuggingFaceEmbeddings(model_name=EMBED_MODEL),   # must match!
)
print("[reopen] count from disk :", len(reopened.get()["ids"]),
      "(matches the build count above)")

probe = reopened.similarity_search("curious little girl", k=2)
preview = probe[0].page_content[:80].replace("\n", " ")
print("[reopen] top hit preview :", preview, "...")


### Scored search: distances, not similarities

`similarity_search_with_score` pairs each hit with its raw distance. In
Chroma's default space LOWER means MORE similar - it is a distance, so do not
read it as a 0-to-1 cosine score. Absolute values also shift between
embedders and distance strategies; treat them as RANKING signals, and set
absolute cutoffs only after eyeballing your own data.

In [5]:
QUERY = "What did the white rabbit wear?"

scored_hits = reopened.similarity_search_with_score(QUERY, k=3)
for rank, (doc, distance) in enumerate(scored_hits, start=1):
    snippet = doc.page_content[:70].replace("\n", " ")
    print(f"{rank}. distance={distance:7.3f} | chapter={doc.metadata['chapter']}")
    print(f"   {snippet} ...")

best = scored_hits[0][1]
worst = scored_hits[-1][1]
print("\nlower distance = more similar (%.3f beats %.3f)" % (best, worst))

1. distance=  0.786 | chapter=2
   After a time she heard a little pattering of feet in the distance, and ...
2. distance=  0.865 | chapter=2
   As she said this she looked down at her hands, and was surprised to se ...
3. distance=  0.895 | chapter=2
   so desperate that she was ready to ask help of any one; so, when the R ...

lower distance = more similar (0.786 beats 0.895)


### Metadata filtering: the payoff of ingest-time tags

Because every chunk carries `chapter`, we can constrain search to one chapter
with a `where` filter - Chroma applies it DURING retrieval, not afterwards.
Classic production uses: per-user scoping, date ranges, language tags. Here:
ask about late-book events but force retrieval back into Chapter 1.

In [6]:
LATE_BOOK_QUERY = "the queen wants to cut off heads"    # late-chapter flavor

plain_hits = reopened.similarity_search(LATE_BOOK_QUERY, k=3)
filtered_hits = reopened.similarity_search(
    LATE_BOOK_QUERY,
    k=3,
    filter={"chapter": "1"},     # where-filter: hard constraint, not a hint
)

plain_chapters = [doc.metadata["chapter"] for doc in plain_hits]
filtered_chapters = [doc.metadata["chapter"] for doc in filtered_hits]

print("UNFILTERED chapters :", plain_chapters)
print("chapter-1 ONLY      :", filtered_chapters)
first_filtered = filtered_hits[0].page_content[:60].replace("\n", " ")
print("first filtered hit  :", first_filtered, "...")

UNFILTERED chapters : ['2', '1', '1']
chapter-1 ONLY      : ['1', '1', '1']
first filtered hit  : Presently she began again. “I wonder if I shall fall right _ ...


### Takeaway

- `from_documents` = embed + insert + persist in one call;
  `count()` is your sanity check after every write.
- Persistence needs THREE matching pieces: path, collection name, embedder.
- Scores from `_with_score` are DISTANCES - lower is better, never a percentage.
- Metadata added at ingest becomes query-time power tools (`filter={...}`).

### Summary

We chunked 20k characters of Alice into chapter-tagged pieces, embedded them
into a persistent Chroma collection, proved a fresh instance reloads it from
disk, inspected ranked hits with their distances, and showed a where-filter
pinning retrieval to Chapter 1. Next: FAISS, the speed-first alternative that
makes YOU handle persistence explicitly.